# Código FT para CV y extracción de métricas

| Variable           | Qué es                                                     | Qué representa          |
| ------------------ | ---------------------------------------------------------- | ----------------------------------------- |
| `time`             | Tiempo de cada muestra de la señal                         | Evolución temporal del movimiento         |
| `dist_raw`         | Distancia euclidiana cruda entre pulgar e índice           | Apertura real de los dedos en píxeles     |
| `amp_raw`          | Distancia normalizada respecto a la máxima amplitud        | Amplitud relativa del movimiento (%)      |
| `dist_smooth`      | Distancia cruda suavizada con Savitzky-Golay               | Movimiento con menos ruido y jitter       |
| `amp_smooth`       | Amplitud normalizada y suavizada                           | Señal biomecánica principal para análisis |
| `tap_count`        | Número de picos detectados                                 | Cantidad de taps realizados               |
| `mean_amp`         | Promedio de amplitudes máximas                             | Tamaño promedio del movimiento            |
| `std_amp`          | Desviación estándar de amplitudes                          | Variabilidad de amplitud entre taps       |
| `cv_amp`           | Coeficiente de variación de amplitud                       | Irregularidad relativa de amplitud        |
| `diff3`            | Diferencia entre amplitud del tap 1 y tap 3                | Decremento temprano de amplitud           |
| `diff5`            | Diferencia entre amplitud del tap 1 y tap 5                | Fatiga motora progresiva                  |
| `diff7`            | Diferencia entre amplitud del tap 1 y tap 7                | Reducción sostenida de amplitud           |
| `diff10`           | Diferencia entre amplitud del tap 1 y tap 10               | Fatiga extrema o decremento tardío        |
| `ft_simple`        | Frecuencia calculada como taps/tiempo total                | Velocidad global del tapping              |
| `ft_iti`           | Frecuencia basada en intervalo inter-tap promedio          | Ritmo promedio del movimiento             |
| `mean_iti`         | Promedio del intervalo entre taps consecutivos             | Tiempo medio entre movimientos            |
| `std_iti`          | Desviación estándar de los ITI                             | Variabilidad temporal                     |
| `cv_iti`           | Coeficiente de variación del ITI                           | Irregularidad temporal relativa           |
| `mean_velocity`    | Velocidad pico promedio normalizada                        | Rapidez promedio del movimiento           |
| `max_velocity`     | Máxima velocidad pico normalizada                          | Movimiento más rápido alcanzado           |
| `opening_velocity` | Velocidad promedio durante apertura de dedos               | Rapidez de extensión                      |
| `closing_velocity` | Velocidad promedio durante cierre de dedos                 | Rapidez de flexión                        |
| `slope_amplitude`  | Pendiente lineal de amplitudes sucesivas                   | Tendencia de fatiga o decremento motor    |
| `peaks`            | Índices de máximos detectados                              | Instantes de máxima apertura              |
| `troughs`          | Índices de mínimos detectados                              | Instantes de cierre                       |
| `freqs`            | Vector de frecuencias FFT                                  | Componentes frecuenciales del movimiento  |
| `spectrum`         | Potencia espectral FFT                                     | Distribución energética de frecuencias    |
| `dominant_freq`    | Frecuencia con máxima energía                              | Ritmo dominante del tapping               |
| `total_energy`     | Energía total del espectro                                 | Intensidad global del movimiento          |
| `regularity_index` | Proporción de energía alrededor de la frecuencia dominante | Regularidad y periodicidad motora         |
| `spectral_entropy` | Entropía del espectro de potencia                          | Complejidad e irregularidad frecuencial   |


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd
from scipy.signal import find_peaks

from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from scipy.stats import linregress
from scipy.stats import entropy

# 1️. INFORMACIÓN DEL VIDEO -------------------------------------------------
def get_video_info(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError("Error abriendo el video")

    fps_metadata = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps_metadata if fps_metadata > 0 else 0

    print("\n===== INFORMACIÓN DEL VIDEO =====")
    print(f"FPS (metadata): {fps_metadata}")
    print(f"Total frames: {total_frames}")
    print(f"Duración estimada (s): {duration:.3f}")

    return cap, fps_metadata


# 2️. EXTRACCIÓN DE LANDMARKS -------------------------------------------------------
def extract_landmarks(cap):
    mpHands = mp.solutions.hands
    allList = []
    timestamps_fps = []

    with mpHands.Hands(
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:

        while cap.isOpened():
            success, img = cap.read()
            if not success:
                break

            ts = cap.get(cv2.CAP_PROP_POS_MSEC)
            timestamps_fps.append(ts / 1000)

            imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = hands.process(imgRGB)

            lmList = []

            if results.multi_hand_landmarks:
                handLms = results.multi_hand_landmarks[0]

                for id, lm in enumerate(handLms.landmark):
                    if id == 4 or id == 8:
                        h, w, c = img.shape
                        cx, cy = int(lm.x * w), int(lm.y * h)
                        lmList.append([id, cx, cy, ts])

            if len(lmList) == 2:
                allList.append(lmList)

    cap.release()
    return np.array(allList), np.array(timestamps_fps)


# 3. ANÁLISIS FPS REAL -------------------------------------------------
def analyze_real_fps(timestamps):
    dt = np.diff(timestamps)

    if len(dt) == 0:
        print("No se pudo calcular FPS real.")
        return None

    fps_real = 1 / dt
    
    mean_dt = np.mean(dt)
    std_dt = np.std(dt)
    cv = std_dt / mean_dt

    print("\n===== ANÁLISIS DE FPS REAL =====")
    print(f"FPS promedio real: {np.mean(fps_real):.2f}")
    print(f"Desviación estándar FPS: {np.std(fps_real):.2f}")
    print(f"FPS mínimo: {np.min(fps_real):.2f}")
    print(f"FPS máximo: {np.max(fps_real):.2f}")
    print(f"CV del Δt: {cv:.4f}")

    return fps_real


# 4️. CONSTRUIR DATAFRAME DE DISTANCIAS ---------------------------------------------------------
def build_dataframe(np_allList):
    length = []

    for i in range(np_allList.shape[0]):
        x1, y1 = np_allList[i][0][1], np_allList[i][0][2]
        x2, y2 = np_allList[i][1][1], np_allList[i][1][2]
        ts = np_allList[i][0][3]

        dist = math.hypot(x2 - x1, y2 - y1)
        length.append([ts, dist])

    df = pd.DataFrame(length, columns=['time', 'dist'])

    # eliminar timestamps duplicados
    df['diff'] = df['time'].diff()
    df = df[df['diff'] > 0]
    df.drop(columns=['diff'], inplace=True)

    if df.empty:
        raise ValueError("DataFrame vacío después de limpiar timestamps.")

    # Normalización
    df['amp'] = (df['dist'] / df['dist'].max()) * 100
    #df.drop(columns=['dist'], inplace=True)

    # Limitar a primeros 20 segundos ---> MODIFICAR ESTO SI EL VIDEO ES MÁS LARGO
    df = df[df['time'] <= 20000]

    return df


# 5. PRE-PREOCESAMIENTO DE LA SEÑAL -----------------------------------------
def preprocess_signal(
    df,
    polyorder=3,
    interpolation_threshold=0.02
):

    df = df.copy()

    # SEÑALES CRUDAS

    t = df['time'].values / 1000.0

    dist_raw = df['dist'].values

    amp_raw = df['amp'].values

    # ESTABILIDAD TEMPORAL

    dt = np.diff(t)

    mean_dt = np.mean(dt)

    std_dt = np.std(dt)

    cv = std_dt / mean_dt

    print("\n===== ESTABILIDAD TEMPORAL =====")

    print(f"Mean dt: {mean_dt:.6f}")
    print(f"STD dt: {std_dt:.6f}")
    print(f"CV: {cv:.4f}")

    # INTERPOLACIÓN
    if cv > interpolation_threshold:

        print("⚠ Muestreo irregular detectado → aplicando interpolación")

        fs_uniform = 1 / mean_dt

        t_uniform = np.arange(
            t[0],
            t[-1],
            1/fs_uniform
        )

        interp_dist = interp1d(
            t,
            dist_raw,
            kind='linear',
            fill_value="extrapolate"
        )

        interp_amp = interp1d(
            t,
            amp_raw,
            kind='linear',
            fill_value="extrapolate"
        )

        dist_raw = interp_dist(t_uniform)

        amp_raw = interp_amp(t_uniform)

        t = t_uniform

    else:

        print("Muestreo suficientemente uniforme → no se interpola")

    # WINDOW SAVITZKY-GOLAY
    window_length = 7
    print(f"Window length Savitzky-Golay: {window_length}")


    # SUAVIZADO
    if len(amp_raw) >= window_length:

        amp_smooth = savgol_filter(
            amp_raw,
            window_length,
            polyorder
        )

        dist_smooth = savgol_filter(
            dist_raw,
            window_length,
            polyorder
        )

    else:
        amp_smooth = amp_raw
        dist_smooth = dist_raw

    # DATAFRAME FINAL
    df_processed = pd.DataFrame({
        'time': t * 1000,
        'dist_raw': dist_raw,
        'amp_raw': amp_raw,
        'dist_smooth': dist_smooth,
        'amp_smooth': amp_smooth
    })

    return df_processed


# 6. CÁLCULO DE MÉTRICAS --------------------------------------------
def compute_temporal_metrics(df):

    signal = df['amp_smooth'].values
    time_sec = df['time'].values / 1000.0

    # PROMINENCE ADAPTATIVO
    prominence = 0.15 * (np.max(signal) - np.min(signal))

    peaks, _ = find_peaks(
        signal,
        prominence=prominence
    )

    troughs, _ = find_peaks(
        -signal,
        prominence=prominence
    )

    # AMPLITUDES
    amps = signal[peaks]

    tap_count = len(amps)

    mean_amp = np.mean(amps) if tap_count > 0 else 0

    std_amp = np.std(amps) if tap_count > 0 else 0

    cv_amp = (
        std_amp / mean_amp
        if mean_amp > 0 else 0
    )

    # DECREMENTO DE AMPLITUD
    def safe_diff(amps, idx):

        if len(amps) > idx:
            return amps[0] - amps[idx]
        else:
            return np.nan

    diff3 = safe_diff(amps, 2)
    diff5 = safe_diff(amps, 4)
    diff7 = safe_diff(amps, 6)
    diff10 = safe_diff(amps, 9)

    # FRECUENCIA E ITI
    total_time = (
        time_sec[-1] - time_sec[0]
        if len(time_sec) > 1 else 0
    )

    ft_simple = (
        tap_count / total_time
        if total_time > 0 else 0
    )

    if tap_count > 1:

        iti = np.diff(time_sec[peaks])

        mean_iti = np.mean(iti)

        std_iti = np.std(iti)

        cv_iti = (
            std_iti / mean_iti
            if mean_iti > 0 else 0
        )

        ft_iti = (
            1 / mean_iti
            if mean_iti > 0 else 0
        )

    else:

        mean_iti = 0
        std_iti = 0
        cv_iti = 0
        ft_iti = 0


    # VELOCIDAD PICO NORMALIZADA
    if tap_count > 1:

        velocity = np.diff(signal) / np.diff(time_sec)

        peak_velocities = []

        for i in range(len(peaks) - 1):

            start = peaks[i]
            end = peaks[i + 1]

            segment = velocity[start:end]

            if len(segment) > 0:

                peak_velocities.append(
                    np.max(np.abs(segment))
                )

        if len(peak_velocities) > 0:

            # Velocidad pico promedio
            mean_velocity = np.mean(peak_velocities)

            # Velocidad máxima global
            max_velocity = np.max(peak_velocities)

            # NORMALIZACIÓN 
            signal_range = (
                np.max(signal) - np.min(signal)
            )

            if signal_range > 0:

                mean_velocity = (
                    mean_velocity / signal_range
                )

                max_velocity = (
                    max_velocity / signal_range
                )

        else:

            mean_velocity = 0
            max_velocity = 0

        # APERTURA vs CIERRE
        positive_vel = velocity[velocity > 0]

        negative_vel = velocity[velocity < 0]

        opening_velocity = (
            np.mean(positive_vel) / signal_range
            if len(positive_vel) > 0 and signal_range > 0
            else 0
        )

        closing_velocity = (
            np.mean(np.abs(negative_vel)) / signal_range
            if len(negative_vel) > 0 and signal_range > 0
            else 0
        )

    else:

        mean_velocity = 0
        max_velocity = 0
        opening_velocity = 0
        closing_velocity = 0

    # FATIGA / DECREMENTO GLOBAL
    if tap_count > 1:

        slope, _, _, _, _ = linregress(
            range(tap_count),
            amps
        )

    else:

        slope = 0

    return {

        "tap_count": tap_count,

        "mean_amp": mean_amp,
        "std_amp": std_amp,
        "cv_amp": cv_amp,

        "diff3": diff3,
        "diff5": diff5,
        "diff7": diff7,
        "diff10": diff10,

        "ft_simple": ft_simple,
        "ft_iti": ft_iti,

        "mean_iti": mean_iti,
        "std_iti": std_iti,
        "cv_iti": cv_iti,

        "mean_velocity": mean_velocity,
        "max_velocity": max_velocity,

        "opening_velocity": opening_velocity,
        "closing_velocity": closing_velocity,

        "slope_amplitude": slope,

        "peaks": peaks,
        "troughs": troughs
    }


# 7. ANÁLISIS FOURIER -------------------------------------------------
def compute_fft(df):

    t = df['time'].values / 1000.0

    signal = df['amp_smooth'].values

    dt = np.mean(np.diff(t))

    fs = 1 / dt

    signal_detrended = signal - np.mean(signal)

    N = len(signal_detrended)

    fft_vals = np.fft.fft(signal_detrended)

    fft_freq = np.fft.fftfreq(N, d=dt)

    positive = fft_freq > 0

    freqs = fft_freq[positive]

    spectrum = (np.abs(fft_vals[positive])**2) / N

    dominant_freq = freqs[np.argmax(spectrum)]

    total_energy = np.sum(spectrum)

    # REGULARIDAD
    band_width = 0.5

    band_mask = (
        (freqs >= dominant_freq - band_width) &
        (freqs <= dominant_freq + band_width)
    )

    energy_f0 = np.sum(spectrum[band_mask])

    regularity_index = (
        energy_f0 / total_energy
        if total_energy > 0 else 0
    )

    # SPECTRAL ENTROPY
    power_norm = spectrum / np.sum(spectrum)

    spectral_entropy = entropy(power_norm)

    return {

        "freqs": freqs,
        "spectrum": spectrum,

        "dominant_freq": dominant_freq,

        "total_energy": total_energy,

        "regularity_index": regularity_index,

        "spectral_entropy": spectral_entropy
    }


# 8. GRÁFICAS -------------------------------------------------

def plot_signal(df, peaks, troughs, hand_label):

    plt.figure(figsize=(10,5))
    plt.plot(df['time'], df['amp_smooth'])
    plt.plot(df['time'].iloc[peaks], df['amp_smooth'].iloc[peaks], "o")
    plt.plot(df['time'].iloc[troughs], df['amp_smooth'].iloc[troughs],"x")
    plt.xlabel("Tiempo (msec)")
    plt.ylabel("Amplitud (%)")
    plt.title(f"Finger Tapping - {hand_label}")
    plt.grid(True)
    plt.show()


def plot_spectrum(freqs, spectrum, hand_label):

    plt.figure(figsize=(10,5))
    plt.plot(freqs, spectrum)
    plt.xlabel("Frecuencia (Hz)")
    plt.ylabel("Magnitud")
    plt.title(f"Espectro de Frecuencia - {hand_label}")
    plt.grid(True)
    plt.show()


# PROCESAR UN VIDEO ---------------------------------------------------------

def analyze_fingertap(video_path):
    
    cap, _ = get_video_info(video_path)
    
    landmarks, timestamps = extract_landmarks(cap)
    analyze_real_fps(timestamps)

    if len(landmarks) == 0:
        raise ValueError("No se detectaron manos")

    df = build_dataframe(landmarks)
    df_processed = preprocess_signal(df)

    temporal_results = compute_temporal_metrics(df_processed)
    fft_results = compute_fft(df_processed)

    # Extraer peaks y troughs fuera del dict
    peaks = temporal_results.pop("peaks")
    troughs = temporal_results.pop("troughs")

    if fft_results is not None:
        temporal_results.update({
            "dominant_freq": fft_results["dominant_freq"],
            "total_energy": fft_results["total_energy"],
            "regularity_index": fft_results["regularity_index"]
        })

    return {
        "df": df_processed,
        "metrics": temporal_results,
        "peaks": peaks,
        "troughs": troughs,
        "freqs": fft_results["freqs"] if fft_results else None,
        "spectrum": fft_results["spectrum"] if fft_results else None
    }

# Análisis de videos

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np


# CONFIGURACIÓN GENERAL
VIDEOS_FOLDER = "videos"
DIAGNOSTIC_CSV = "fis_diagnostic.csv"
OUTPUT_METRICS = "ALL_METRICS.csv"
SIGNALS_FOLDER = "signals"

os.makedirs(SIGNALS_FOLDER, exist_ok=True)

# LEER CSV DIAGNÓSTICO
diagnostic_df = pd.read_csv(DIAGNOSTIC_CSV)


# DATAFRAME GLOBAL MÉTRICAS
all_metrics = []


# RECORRER TODOS LOS VIDEOS
for idx, row in diagnostic_df.iterrows():

    # DATOS DEL CSV
    video_id = row["ID"]
    updrs = row["UPDRS"]

    # EXTRAER MANO DESDE EL ID
    if "DCHA" in video_id:
        mano = "DCHA"

    elif "IZDA" in video_id:
        mano = "IZDA"

    else:
        mano = "UNKNOWN"

    # BUSCAR VIDEO
    video_path = None
    possible_extensions = [".mp4", ".avi", ".mov", ".mkv"]

    for ext in possible_extensions:

        temp_path = os.path.join(
            VIDEOS_FOLDER,
            video_id + ext
        )

        if os.path.exists(temp_path):
            video_path = temp_path
            break

    # SI NO EXISTE VIDEO
    if video_path is None:
        print(f"\n Video no encontrado: {video_id}")
        continue

    print("\n================================================")
    print(f"Procesando: {video_id}")
    print("================================================")

    # ANALIZAR VIDEO
    try:
        results = analyze_fingertap(video_path)

    except Exception as e:
        print(f"Error procesando {video_id}")
        print(e)
        continue


    # OBTENER DATAFRAME SEÑAL
    df_signal = results["df"]

    # GUARDAR CSV DE SEÑAL INDIVIDUAL
    signal_output_path = os.path.join(
        SIGNALS_FOLDER,
        f"{video_id}.csv"
    )

    df_signal.to_csv(
        signal_output_path,
        index=False
    )

    print(f"Señal guardada: {signal_output_path}")

    # OBTENER MÉTRICAS
    metrics = results["metrics"]

    # CREAR FILA MÉTRICAS
    metrics_row = {
        "ID": video_id,
        "UPDRS": updrs,
        "MANO": mano
    }

    # Agregar todas las métricas automáticamente
    metrics_row.update(metrics)

    # AGREGAR A LISTA GLOBAL
    all_metrics.append(metrics_row)


# GUARDAR ALL_METRICS
metrics_df = pd.DataFrame(all_metrics)

metrics_df.to_csv(
    OUTPUT_METRICS,
    index=False
)

print("\n==========================================")
print("ALL_METRICS.csv generado correctamente")
print("==========================================")